In [1]:
# =========================
# qcqp_generator_mixed_v2.py
#
# Updates (targeted to cover SDP-better-but-predicted-LP failure patterns):
#  1) Add non-bipartite / odd-cycle-rich sparsity generator ("oddcycle") to create bounded+hollow cases
#     where SDP is more likely to dominate McCormick/LP relaxations.
#  2) Add bilinear shared-support mode to increase cross-constraint coupling (hard for LP, helps SDP).
#  3) Add MOSEK-safe stabilization:
#       - cap spectral radius (|lambda|max) by scaling
#       - avoid near-singularity for NON-bilinear matrices via small diagonal shift (preserves sparsity)
#       - keep bilinear diagonal exactly zero (no diagonal shifts), resample weights if too singular
#  4) Fix single-type sampling for bilinear to sample from bilinear-eligible pool (not all_matrices).
#  5) bounds_exist=True  -> l=0, u=1 (unchanged)
#     bounds_exist=False -> l=-inf, u=+inf; artificial bounds stored as inst.l_art / inst.u_art
#     pd_index now sampled from integers(1, m) so PD constraint is never the objective.
# =========================

import os
import numpy as np
import pandas as pd
from sklearn.datasets import make_spd_matrix
import networkx as nx
from networkx.algorithms import bipartite
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional, Tuple


# -----------------
# RNG helper
# -----------------
def make_rng(seed: int):
    return np.random.default_rng(int(seed))


# -----------------
# Graph -> weighted symmetric matrix helper
# -----------------
def adjacency_to_distance(A, rng: np.random.Generator, allow_zero_edge_weights: bool = False):
    if allow_zero_edge_weights:
        random_weights = rng.integers(-10, 11, size=A.shape)
    else:
        # exclude 0 so an "edge" doesn't become 0 and disappear from sparsity
        random_weights = rng.integers(1, 11, size=A.shape) * rng.choice([-1, 1], size=A.shape)

    D = np.where(A == 1, random_weights, 0)
    D = np.tril(D) + np.triu(D.T, 1)
    np.fill_diagonal(D, rng.integers(1, 11, size=D.shape[0]))
    return D, False


# -----------------
# Existing sparsity generators
# -----------------
def generate_bipartite_distance_matrix(n, rng: np.random.Generator):
    k = int(rng.integers(2, n - 1))
    e = int(rng.integers(2, k * (n - k) + 1))
    seed_int = int(rng.integers(0, 2**31 - 1))
    B = bipartite.gnmk_random_graph(k, n - k, e, seed=seed_int)
    A = nx.to_numpy_array(B, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_tree_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    T = nx.random_tree(n, seed=seed_int)
    A = nx.to_numpy_array(T, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_planar_distance_matrix(n, rng: np.random.Generator):
    G = nx.complete_graph(n)
    edges = rng.permutation(list(G.edges)).tolist()
    is_planar = False
    while not is_planar and edges:
        edge = edges.pop()
        G.remove_edge(*edge)
        is_planar, _ = nx.check_planarity(G)
    if not nx.is_connected(G):
        remaining_edges = rng.permutation(list(nx.non_edges(G))).tolist()
        if remaining_edges:
            G.add_edge(*remaining_edges[0])
    A = nx.to_numpy_array(G, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_chordal_distance_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    T = nx.random_tree(n, seed=seed_int)
    for node in range(n):
        neighbors = list(nx.neighbors(T, node))
        if len(neighbors) >= 2:
            v, w = rng.choice(neighbors, size=2, replace=False)
            T.add_edge(int(v), int(w))
    A = nx.to_numpy_array(T, dtype=int)
    return adjacency_to_distance(A, rng)


# -----------------
# NEW: odd-cycle-rich (non-bipartite) sparsity generator
# -----------------
def generate_oddcycle_distance_matrix(n: int, rng: np.random.Generator):
    """
    Builds a connected, explicitly non-bipartite graph by forcing at least one triangle (odd cycle),
    then converts to a weighted symmetric matrix via adjacency_to_distance.
    """
    n = int(n)
    if n <= 2:
        A = np.zeros((n, n), dtype=int)
        return adjacency_to_distance(A, rng)

    seed_int = int(rng.integers(0, 2**31 - 1))

    # density heuristic: moderate density; higher for small n, lower for larger n
    p = float(np.clip(6.0 / max(n, 6), 0.12, 0.32))
    G = nx.gnp_random_graph(n, p, seed=seed_int, directed=False)

    # ensure connectivity by stitching components
    if not nx.is_connected(G):
        comps = [list(c) for c in nx.connected_components(G)]
        for i in range(len(comps) - 1):
            u = int(rng.choice(comps[i]))
            v = int(rng.choice(comps[i + 1]))
            G.add_edge(u, v)

    # force a triangle to guarantee non-bipartite (odd cycle)
    nodes = np.arange(n)
    a, b, c = rng.choice(nodes, size=3, replace=False)
    G.add_edge(int(a), int(b))
    G.add_edge(int(b), int(c))
    G.add_edge(int(c), int(a))

    A = nx.to_numpy_array(G, dtype=int)
    return adjacency_to_distance(A, rng)


# -----------------
# Dense / spectral generators
# -----------------
def generate_symmetric_matrix(n: int, rng: np.random.Generator):
    A = rng.uniform(-5, 5, size=(n, n))
    D = 0.5 * (A + A.T)
    return D, False


def generate_spd_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    D = make_spd_matrix(n, random_state=seed_int)
    return D, False


def _rand_Q(n, rng):
    Q, _ = np.linalg.qr(rng.uniform(0, 9, size=(n, n)))
    return Q


# ---- SIGN-TUNABLE generators (n_neg controls #negative eigenvalues / diagonal signs) ----
def diag_ordered_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    v[:n_neg] = -1
    return np.diag(v), False


def diag_random_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] = -1
    return np.diag(v), False


def diag_random_ints(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = rng.integers(1, 11, size=n).astype(float)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] *= -1
    return np.diag(v), False


def eigvals_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    v[:n_neg] = -1
    L = np.diag(v)
    Q = _rand_Q(n, rng)
    return Q @ L @ Q.T, False


def eigvals_ints(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = rng.integers(1, 11, size=n).astype(float)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] *= -1
    L = np.diag(v)
    Q = _rand_Q(n, rng)
    return Q @ L @ Q.T, False


# -----------------
# Artificial bounds
# -----------------
def compute_artificial_bounds(A: np.ndarray, b: np.ndarray, c: float):
    """
    Bounds from ellipsoid:
        x^T A x + 2 b^T x + c <= 0
    assuming A is PD.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    n = A.shape[0]

    L = np.linalg.cholesky(A)

    def solve_A(rhs):
        rhs = np.asarray(rhs, dtype=float)
        y = np.linalg.solve(L, rhs)
        x = np.linalg.solve(L.T, y)
        return x

    Ainv_b = solve_A(b)
    b_Ainv_b = float(b @ Ainv_b)
    t = b_Ainv_b - float(c)

    if t < 0:
        raise ValueError("Invalid bounding: b^T A^-1 b - c must be non-negative.")

    sqrt_term = float(np.sqrt(max(t, 0.0)))
    Ainv = solve_A(np.eye(n))
    diag_Ainv = np.diag(Ainv)

    l = np.zeros(n, dtype=float)
    u = np.zeros(n, dtype=float)

    for i in range(n):
        di = float(diag_Ainv[i])
        if di < 0:
            raise ValueError("Numerical issue: negative diagonal entry in A^{-1}.")
        delta = sqrt_term * float(np.sqrt(max(di, 0.0)))
        center = -float(Ainv_b[i])
        l[i] = center - delta
        u[i] = center + delta

    return l, u


# -----------------
# Instance container
# -----------------
@dataclass
class QCQPInstance:
    instance_id: int
    bounds_exist: bool
    A_list: list
    b_list: list
    c_list: list
    l: np.ndarray
    u: np.ndarray
    results: dict = field(default_factory=dict)

    # optional metadata
    n: Optional[int] = None
    m: Optional[int] = None
    group_id: Optional[int] = None
    size_tag: Optional[str] = None

    # artificial bounds stored as flat attributes (matching second code)
    l_art: Optional[np.ndarray] = None
    u_art: Optional[np.ndarray] = None


# -----------------
# Stabilization helpers (MOSEK-safe; preserve sparsity)
# -----------------
def _sym(A: np.ndarray) -> np.ndarray:
    return 0.5 * (A + A.T)


def _eig_abs_max(A: np.ndarray) -> float:
    w = np.linalg.eigvalsh(A)
    return float(np.max(np.abs(w)))


def _eig_abs_min(A: np.ndarray) -> float:
    w = np.linalg.eigvalsh(A)
    return float(np.min(np.abs(w)))


def _scale_to_eig_abs_max(A: np.ndarray, max_abs: float) -> np.ndarray:
    cur = _eig_abs_max(A)
    if cur <= 0:
        return A
    if cur > max_abs:
        return A * (max_abs / cur)
    return A


def _stabilize_spd(A: np.ndarray, eig_min: float, eig_max: float) -> np.ndarray:
    """
    Keep SPD, avoid extreme condition numbers, without densifying beyond what SPD already is.
    """
    A = _sym(np.asarray(A, dtype=float))
    n = A.shape[0]
    w = np.linalg.eigvalsh(A)
    lam_min = float(w[0])

    if lam_min <= 0:
        A = A + (abs(lam_min) + float(eig_min)) * np.eye(n)

    w = np.linalg.eigvalsh(A)
    lam_min = float(w[0])
    if lam_min < eig_min:
        A = A + (eig_min - lam_min) * np.eye(n)

    w = np.linalg.eigvalsh(A)
    lam_max = float(w[-1])
    if lam_max > eig_max:
        A = A * (eig_max / lam_max)

    return _sym(A)


def _stabilize_general_symmetric_sparse(A: np.ndarray, eig_abs_max: float, eig_abs_min: float) -> np.ndarray:
    """
    For non-bilinear matrices:
      - scale to cap |lambda|max
      - if near-singular, apply small diagonal shift to push away from 0 (preserves sparsity)
    """
    A = _sym(np.asarray(A, dtype=float))
    n = A.shape[0]

    A = _scale_to_eig_abs_max(A, eig_abs_max)

    w = np.linalg.eigvalsh(A)
    min_abs = float(np.min(np.abs(w)))
    if min_abs < eig_abs_min:
        delta = (eig_abs_min - min_abs)
        A = A + delta * np.eye(n)
        A = _scale_to_eig_abs_max(A, eig_abs_max)

    return _sym(A)


def _make_bilinear_from_mask(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    n = mask.shape[0]
    rv = rng.uniform(-2, 3, size=(n, n))
    A = np.where(mask != 0, rv, 0.0)
    A = _sym(A)
    np.fill_diagonal(A, 0.0)
    return A


def _bilinear_mask_from_matrix(A: np.ndarray) -> np.ndarray:
    M = (np.asarray(A) != 0).astype(float)
    M = _sym(M)
    np.fill_diagonal(M, 0.0)
    return M


def _stabilize_bilinear(A: np.ndarray, eig_abs_max: float, eig_abs_min: float) -> Optional[np.ndarray]:
    """
    For bilinear: keep diag exactly zero.
    Only scaling allowed; if too close to singular return None to force resampling.
    """
    A = _sym(np.asarray(A, dtype=float))
    np.fill_diagonal(A, 0.0)

    A = _scale_to_eig_abs_max(A, eig_abs_max)
    np.fill_diagonal(A, 0.0)
    A = _sym(A)

    w = np.linalg.eigvalsh(A)
    min_abs = float(np.min(np.abs(w)))
    if min_abs < eig_abs_min:
        return None
    return A


# -----------------
# Main generator
# -----------------
class QCQPInstanceGenerator:
    def __init__(
        self,
        n,
        m,
        num_instances,
        seed,
        outcome_weights,
        matrix_weights=None,
        # MOSEK-safe defaults
        eig_abs_max_general: float = 25.0,
        eig_abs_min_general: float = 1e-4,
        spd_eig_min: float = 0.5,
        spd_eig_max: float = 15.0,
        eig_abs_max_bilinear: float = 10.0,
        eig_abs_min_bilinear: float = 1e-5,
        bilinear_share_support_prob: float = 0.55,
        bilinear_force_nonbipartite_prob: float = 0.65,
        bilinear_boost_oddcycle_weight: float = 4.0,
        max_bilinear_resample_tries: int = 30,
    ):
        self.n = int(n)
        self.m = int(m)  # total: 1 objective + (m-1) constraints
        self.master_seed = int(seed)
        self.num_instances = int(num_instances)
        self.outcome_weights = np.asarray(outcome_weights, dtype=float)
        self.matrix_weights = dict(matrix_weights) if matrix_weights is not None else {}

        # stabilization params
        self.eig_abs_max_general = float(eig_abs_max_general)
        self.eig_abs_min_general = float(eig_abs_min_general)
        self.spd_eig_min = float(spd_eig_min)
        self.spd_eig_max = float(spd_eig_max)
        self.eig_abs_max_bilinear = float(eig_abs_max_bilinear)
        self.eig_abs_min_bilinear = float(eig_abs_min_bilinear)

        # diversity knobs
        self.bilinear_share_support_prob = float(bilinear_share_support_prob)
        self.bilinear_force_nonbipartite_prob = float(bilinear_force_nonbipartite_prob)
        self.bilinear_boost_oddcycle_weight = float(bilinear_boost_oddcycle_weight)
        self.max_bilinear_resample_tries = int(max_bilinear_resample_tries)

        self.cum_weights = np.cumsum(self.outcome_weights)

    def _base_seed(self):
        return int(self.master_seed) + 1_000_003 * int(self.m) + 100_069 * int(self.n)

    def determine_outcome(self, instance_id):
        local_seed = self._base_seed() + int(instance_id) * 13 + 7
        rng = np.random.default_rng(local_seed)
        outcome = rng.choice(
            a=np.arange(1, len(self.outcome_weights) + 1),
            p=self.outcome_weights / np.sum(self.outcome_weights),
        )
        return int(outcome)

    def _weight(self, mt: str) -> float:
        return float(self.matrix_weights.get(mt, 1.0))

    def generate_matrix(self, matrix_type, bilinear, rng, n_neg=None, bilinear_mask: Optional[np.ndarray] = None):
        MATRIX_FUNCS = {
            "bipartite": generate_bipartite_distance_matrix,
            "tree": generate_tree_matrix,
            "planar": generate_planar_distance_matrix,
            "chordal": generate_chordal_distance_matrix,
            "oddcycle": generate_oddcycle_distance_matrix,
            "symmetric": generate_symmetric_matrix,
            "diag_ord1": diag_ordered_ones,
            "diag_rand1": diag_random_ones,
            "diag_randI": diag_random_ints,
            "eig_ones": eigvals_ones,
            "eig_ints": eigvals_ints,
            "spd": generate_spd_matrix,
        }
        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}

        # --- bilinear: build from a mask (shared or per-matrix) and enforce MOSEK-safe spectrum ---
        if bilinear:
            if bilinear_mask is None:
                if matrix_type in SIGN_TUNABLE:
                    A0, _ = MATRIX_FUNCS[matrix_type](self.n, rng, n_neg=n_neg)
                else:
                    A0, _ = MATRIX_FUNCS[matrix_type](self.n, rng)
                bilinear_mask = _bilinear_mask_from_matrix(A0)

            for _ in range(self.max_bilinear_resample_tries):
                A = _make_bilinear_from_mask(bilinear_mask, rng)
                A_stab = _stabilize_bilinear(A, self.eig_abs_max_bilinear, self.eig_abs_min_bilinear)
                if A_stab is not None:
                    return A_stab

            # fallback: return scaled bilinear even if near-singular
            A = _make_bilinear_from_mask(bilinear_mask, rng)
            A = _scale_to_eig_abs_max(A, self.eig_abs_max_bilinear)
            np.fill_diagonal(A, 0.0)
            return _sym(A)

        # --- non-bilinear ---
        if matrix_type in SIGN_TUNABLE:
            A, _ = MATRIX_FUNCS[matrix_type](self.n, rng, n_neg=n_neg)
        else:
            A, _ = MATRIX_FUNCS[matrix_type](self.n, rng)

        A = _sym(np.asarray(A, dtype=float))

        if matrix_type == "spd":
            A = _stabilize_spd(A, self.spd_eig_min, self.spd_eig_max)
        else:
            A = _stabilize_general_symmetric_sparse(A, self.eig_abs_max_general, self.eig_abs_min_general)

        return A

    def generate_instance(self, instance_id):
        base = self._base_seed()
        rng = np.random.default_rng(base + int(instance_id) * 10 + 3)

        outcome = self.determine_outcome(instance_id)
        bounds_exist = outcome > 4
        matrix_combined = outcome in {1, 2, 5, 6}
        bilinear = outcome in {1, 3, 5, 7}

        # Eligible sets
        sparsity_matrices = [
            "bipartite", "tree", "planar", "chordal",
            "oddcycle",
            "symmetric", "eig_ones", "eig_ints"
        ]
        non_eligible_matrices = ["spd", "diag_ord1", "diag_rand1", "diag_randI"]
        all_matrices = sparsity_matrices + non_eligible_matrices

        matrix_types = sparsity_matrices if bilinear else all_matrices

        # If bounded+bilinear, boost odd-cycle generation to diversify SDP-strong cases
        if bounds_exist and bilinear:
            boosted_weights = {mt: self._weight(mt) for mt in matrix_types}
            boosted_weights["oddcycle"] = boosted_weights.get("oddcycle", 1.0) * self.bilinear_boost_oddcycle_weight
        else:
            boosted_weights = None

        if matrix_combined:
            if boosted_weights is not None:
                weights = np.array([boosted_weights[mt] for mt in matrix_types], dtype=float)
            else:
                weights = np.array([self._weight(mt) for mt in matrix_types], dtype=float)
            prob = weights / weights.sum()
            chosen_matrix_types = rng.choice(matrix_types, size=self.m, replace=True, p=prob).tolist()
        else:
            if boosted_weights is not None:
                weights = np.array([boosted_weights[mt] for mt in matrix_types], dtype=float)
            else:
                weights = np.array([self._weight(mt) for mt in matrix_types], dtype=float)
            prob = weights / weights.sum()
            single_type = str(rng.choice(matrix_types, p=prob))
            chosen_matrix_types = [single_type] * self.m

        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}
        num_neg_eigvals = []
        for mt in chosen_matrix_types:
            n_neg = int(rng.integers(0, self.n + 1)) if mt in SIGN_TUNABLE else None
            num_neg_eigvals.append(n_neg)

        # PD injection index — sampled from integers(1, m) so it is always a constraint, not the objective
        pd_index = None
        if not bounds_exist:
            pd_index = int(rng.integers(1, self.m)) if self.m > 1 else 0

        # Shared-support mode for bounded+bilinear
        use_shared_support = bool(bilinear and bounds_exist and (rng.random() < self.bilinear_share_support_prob))
        force_nonbipartite = bool(bilinear and bounds_exist and (rng.random() < self.bilinear_force_nonbipartite_prob))

        shared_mask = None
        if bilinear and use_shared_support:
            base_type = "oddcycle" if force_nonbipartite else str(rng.choice(matrix_types))
            sub_rng0 = np.random.default_rng(base + int(instance_id) * 1000 + 0 * 97 + 11)
            if base_type in SIGN_TUNABLE:
                A0, _ = (diag_ordered_ones(self.n, sub_rng0, n_neg=int(sub_rng0.integers(0, self.n + 1))))
            else:
                A0, _ = {
                    "bipartite": generate_bipartite_distance_matrix,
                    "tree": generate_tree_matrix,
                    "planar": generate_planar_distance_matrix,
                    "chordal": generate_chordal_distance_matrix,
                    "oddcycle": generate_oddcycle_distance_matrix,
                    "symmetric": generate_symmetric_matrix,
                    "eig_ones": eigvals_ones,
                    "eig_ints": eigvals_ints,
                    "spd": generate_spd_matrix,
                    "diag_ord1": diag_ordered_ones,
                    "diag_rand1": diag_random_ones,
                    "diag_randI": diag_random_ints,
                }[base_type](self.n, sub_rng0)
            shared_mask = _bilinear_mask_from_matrix(A0)

        A_list, b_list, c_list = [], [], []

        for i, matrix_type in enumerate(chosen_matrix_types):
            sub_rng = np.random.default_rng(base + int(instance_id) * 1000 + i * 97 + 11)
            n_neg = num_neg_eigvals[i]

            # Inject one PD constraint for bounds_exist=False
            if not bounds_exist and i == pd_index:
                while True:
                    A_pd, _ = generate_spd_matrix(self.n, sub_rng)
                    A_pd = _stabilize_spd(A_pd, self.spd_eig_min, self.spd_eig_max)
                    b_pd = sub_rng.uniform(-5, 5, size=self.n)
                    c_pd = -sub_rng.uniform(0, 10)
                    try:
                        np.linalg.cholesky(A_pd)
                        _ = compute_artificial_bounds(A_pd, b_pd, c_pd)
                        break
                    except Exception:
                        continue
                A_list.append(A_pd)
                b_list.append(b_pd)
                c_list.append(c_pd)
                continue

            # For bounded+bilinear, optionally force non-bipartite by switching type to oddcycle
            effective_type = "oddcycle" if (bilinear and bounds_exist and force_nonbipartite) else matrix_type

            A = self.generate_matrix(
                effective_type,
                bilinear,
                sub_rng,
                n_neg=n_neg,
                bilinear_mask=shared_mask if (bilinear and use_shared_support) else None,
            )
            A_list.append(A)
            b_list.append(sub_rng.uniform(-5, 5, size=self.n))
            c_list.append(-sub_rng.uniform(0, 10))

        # ── BOUNDS (matching second code) ────────────────────────────────────
        # bounds_exist=True  -> l=0, u=1
        # bounds_exist=False -> l=-inf, u=+inf; artificial bounds in l_art/u_art
        if bounds_exist:
            l = np.zeros(self.n, dtype=float)
            u = np.ones(self.n, dtype=float)
        else:
            l = np.full(self.n, -np.inf, dtype=float)
            u = np.full(self.n,  np.inf, dtype=float)

        inst = QCQPInstance(int(instance_id), bool(bounds_exist), A_list, b_list, c_list, l, u)
        inst.n = self.n
        inst.m = self.m
        inst.l_art = None
        inst.u_art = None

        # Compute and store artificial bounds as flat attributes (not nested in results)
        if not bounds_exist and pd_index is not None:
            try:
                l_art, u_art = compute_artificial_bounds(
                    A_list[pd_index], b_list[pd_index], c_list[pd_index]
                )
                inst.l_art = l_art
                inst.u_art = u_art
            except Exception:
                inst.l_art = None
                inst.u_art = None

        return inst

    def save_instances(self, filename, start_id=0):
        instances = [self.generate_instance(i) for i in range(int(start_id), int(start_id) + self.num_instances)]
        pd.to_pickle(instances, filename)


# -----------------
# Mixed-size PKL generator
# -----------------
class QCQPSizeMixtureGenerator:
    """
    size_plan: list of dicts, each like:
        {"n": 5, "m_constr": 10, "count": 10}  # m_total = m_constr + 1
    Optional per-group overrides:
        {"n":..., "m_constr":..., "count":..., "outcome_weights":..., "matrix_weights":..., **kwargs}
    """

    def __init__(
        self,
        size_plan: List[Dict[str, Any]],
        seed: int,
        default_outcome_weights: np.ndarray,
        default_matrix_weights: Dict[str, float],
        shuffle: bool = True,
        shuffle_seed_offset: int = 12345,
        generator_defaults: Optional[Dict[str, Any]] = None,
    ):
        self.master_seed = int(seed)
        self.size_plan = list(size_plan)
        self.default_outcome_weights = np.asarray(default_outcome_weights, dtype=float)
        self.default_matrix_weights = dict(default_matrix_weights)
        self.generator_defaults = dict(generator_defaults) if generator_defaults is not None else {}

        self._gens: Dict[int, QCQPInstanceGenerator] = {}
        self._group_meta: Dict[int, Tuple[int, int, int]] = {}

        jobs: List[Tuple[int, int]] = []

        for gid, spec in enumerate(self.size_plan):
            n = int(spec["n"])
            m_constr = int(spec["m_constr"])
            count = int(spec["count"])
            m_total = m_constr + 1

            ow = np.asarray(spec.get("outcome_weights", self.default_outcome_weights), dtype=float)
            mw = spec.get("matrix_weights", self.default_matrix_weights)

            per_group_kwargs = dict(self.generator_defaults)
            for k, v in spec.items():
                if k in {
                    "eig_abs_max_general", "eig_abs_min_general",
                    "spd_eig_min", "spd_eig_max",
                    "eig_abs_max_bilinear", "eig_abs_min_bilinear",
                    "bilinear_share_support_prob", "bilinear_force_nonbipartite_prob",
                    "bilinear_boost_oddcycle_weight", "max_bilinear_resample_tries",
                }:
                    per_group_kwargs[k] = v

            gen = QCQPInstanceGenerator(
                n=n,
                m=m_total,
                num_instances=count,
                seed=self.master_seed,
                outcome_weights=ow,
                matrix_weights=mw,
                **per_group_kwargs,
            )
            self._gens[gid] = gen
            self._group_meta[gid] = (n, m_total, count)

            for local_id in range(count):
                jobs.append((gid, local_id))

        if shuffle:
            rng = np.random.default_rng(self.master_seed + int(shuffle_seed_offset))
            perm = rng.permutation(len(jobs))
            self.jobs = [jobs[i] for i in perm]
        else:
            self.jobs = jobs

        self.num_instances = len(self.jobs)

    def generate_instance(self, global_id: int) -> QCQPInstance:
        gid, local_id = self.jobs[int(global_id)]
        inst = self._gens[gid].generate_instance(local_id)

        inst.instance_id = int(global_id)
        inst.group_id = int(gid)
        inst.n = inst.l.size
        inst.m = len(inst.A_list)
        inst.size_tag = f"n{inst.n}_m{inst.m}"

        return inst

    def save_instances(self, filename: str, start_id: int = 0):
        start_id = int(start_id)
        instances = [self.generate_instance(i) for i in range(start_id, start_id + self.num_instances)]
        pd.to_pickle(instances, filename)

In [2]:
# =========================
# CELL 1 — BUILD + SAVE MIXED-SIZE PKL
# =========================
import os
import numpy as np
import pandas as pd

if __name__ == "__main__":
    custom_matrix_weights = {
        "bipartite": 1,
        "tree": 1,
        "planar": 1,
        "chordal": 1,
        "oddcycle": 1,
        "symmetric": 1,
        "spd": 10,
        "diag_ord1": 1,
        "diag_rand1": 1,
        "diag_randI": 1,
        "eig_ones": 1,
        "eig_ints": 1,
    }

    outcome_weights = np.array(
        [0.08, 0.05, 0.07, 0.05, 0.20, 0.10, 0.25, 0.20],
        dtype=float
    )

    size_plan = [
        {'n': 10, 'm_constr': 18, 'count': 400},
        {'n': 9,  'm_constr': 10, 'count': 400},
        {'n': 16, 'm_constr': 48, 'count': 400},
        {'n': 43, 'm_constr': 84, 'count': 400},
        {'n': 14, 'm_constr': 28, 'count': 400},
        {'n': 33, 'm_constr': 71, 'count': 400},
        {'n': 18, 'm_constr': 56, 'count': 400},
        {'n': 4,  'm_constr': 2,  'count': 400},
        {'n': 14, 'm_constr': 36, 'count': 400},
        {'n': 14, 'm_constr': 26, 'count': 400},
        {'n': 14, 'm_constr': 22, 'count': 400},
        {'n': 12, 'm_constr': 16, 'count': 400},
        {'n': 17, 'm_constr': 36, 'count': 400},
        {'n': 12, 'm_constr': 26, 'count': 400},
        {'n': 14, 'm_constr': 23, 'count': 400},
        {'n': 19, 'm_constr': 25, 'count': 400},
        {'n': 33, 'm_constr': 79, 'count': 400},
        {'n': 10, 'm_constr': 12, 'count': 400},
        {'n': 5,  'm_constr': 3,  'count': 400},
        {'n': 18, 'm_constr': 37, 'count': 400},
    ]

    gen = QCQPSizeMixtureGenerator(
        size_plan=size_plan,
        seed=42,
        default_outcome_weights=outcome_weights,
        default_matrix_weights=custom_matrix_weights,
        shuffle=True,
        generator_defaults={
            "bilinear_share_support_prob":       0.55,
            "bilinear_force_nonbipartite_prob":  0.35,
            "bilinear_boost_oddcycle_weight":    3.0,
            "eig_abs_max_general":               25.0,
            "eig_abs_min_general":               1e-4,
            "spd_eig_min":                       0.5,
            "spd_eig_max":                       15.0,
            "eig_abs_max_bilinear":              10.0,
            "eig_abs_min_bilinear":              1e-5,
            "max_bilinear_resample_tries":       30,
        },
    )

    pkl_path = "instances_synthetic.pkl"
    gen.save_instances(pkl_path)
    print("✅ saved:", pkl_path, " | num_instances =", gen.num_instances)

✅ saved: instances_synthetic.pkl  | num_instances = 8000


In [3]:
# =========================
# CELL 2 — LOAD PKL + SOLVE + WRITE CSV
# =========================

import os
import gc
import numpy as np
import pandas as pd
import psutil
import pickle

from qcqp_solversdg import QCQPSolver


# -----------------
# Bounds helpers (matching second code)
# -----------------
def _has_finite_bounds(l, u):
    l = np.asarray(l, dtype=float).reshape(-1)
    u = np.asarray(u, dtype=float).reshape(-1)
    return (
        l.size > 0
        and u.size == l.size
        and np.all(np.isfinite(l))
        and np.all(np.isfinite(u))
    )


def _bounds_valid(l, u):
    l = np.asarray(l, dtype=float).reshape(-1)
    u = np.asarray(u, dtype=float).reshape(-1)
    if l.size == 0 or u.size == 0 or l.size != u.size:
        return False
    if not (np.all(np.isfinite(l)) and np.all(np.isfinite(u))):
        return False
    if not np.all(u > l):
        return False
    return True


def _working_bounds_for_solve(inst_d):
    """
    Use original bounds if finite (bounds_exist=True).
    Otherwise fall back to cached artificial bounds (l_art / u_art).
    Raises RuntimeError if neither is available.
    """
    l = np.asarray(inst_d.get("l", []), dtype=float).reshape(-1)
    u = np.asarray(inst_d.get("u", []), dtype=float).reshape(-1)
    bounds_exist = bool(inst_d.get("bounds_exist", False))

    if bounds_exist and _has_finite_bounds(l, u):
        return l, u

    # Fall back to artificial bounds stored as flat attributes
    l_art = inst_d.get("l_art", None)
    u_art = inst_d.get("u_art", None)
    if l_art is not None and u_art is not None and _bounds_valid(l_art, u_art):
        return np.asarray(l_art, dtype=float).reshape(-1), np.asarray(u_art, dtype=float).reshape(-1)

    raise RuntimeError(
        "Missing l_art/u_art for bounds_exist=False instance "
        "(should not happen with redesigned generator)."
    )


# -----------------
# Main solver loop
# -----------------
def run_instances_from_pkl(
    pkl_path,
    out_csv="instancesobj_synthetic.csv",
    solve_lp=True,
    solve_sdp=True,
    solve_sdp_prime=True,
    solve_sdp_rlt=True,
    print_every=10,
    gc_every=200,
    solver_kwargs=None,
):
    if solver_kwargs is None:
        solver_kwargs = {}

    with open(pkl_path, "rb") as f:
        instances = pickle.load(f)

    if out_csv and os.path.exists(out_csv):
        os.remove(out_csv)

    header_written = False
    proc = psutil.Process(os.getpid())

    split = ""
    if out_csv:
        if "tst" in out_csv:
            split = "test"
        elif "tr" in out_csv or "train" in out_csv:
            split = "train"

    for idx, inst in enumerate(instances):
        try:
            inst_d = inst.__dict__ if hasattr(inst, "__dict__") else inst

            A_list = inst_d["A_list"]
            b_list = inst_d["b_list"]
            c_list = inst_d["c_list"]

            l_work, u_work = _working_bounds_for_solve(inst_d)

            n        = int(inst_d.get("n", None) or l_work.size or len(b_list[0]))
            m_total  = int(inst_d.get("m", None) or len(A_list))
            m_constr = m_total - 1

            solver = QCQPSolver(A_list, b_list, c_list, l_work, u_work, **solver_kwargs)

            lp  = solver.solve("LP")  if solve_lp  else {"objective": np.nan, "time": np.nan, "status": "skipped"}
            sdp = solver.solve("SDP") if solve_sdp else {"objective": np.nan, "time": np.nan, "status": "skipped"}

            sdp_p = (
                solver.solve("SDP_PRIME")
                if solve_sdp_prime
                else {"objective": np.nan, "time": np.nan, "status": "skipped"}
            )
            sdp_rlt = (
                solver.solve("SDP_RLT")
                if solve_sdp_rlt
                else {"objective": np.nan, "time": np.nan, "status": "skipped"}
            )

            if print_every and (idx % print_every == 0):
                print(
                    f"idx={idx} id={inst_d.get('instance_id', idx)} | "
                    f"n={n} m_constr={m_constr} | "
                    f"LP={lp.get('objective')} | SDP={sdp.get('objective')}",
                    flush=True,
                )

            row = {
                "id":             int(inst_d.get("instance_id", idx)),
                "split":          split,
                "seed":           inst_d.get("seed", None),
                "id_global":      inst_d.get(
                                      "id_global",
                                      f"{inst_d.get('instance_id', idx)}_n{n}_mc{m_constr}"
                                  ),
                "group_id":       inst_d.get("group_id", None),
                "size_tag":       inst_d.get("size_tag", None),
                "bounds_exist":   inst_d.get("bounds_exist", None),
                "n":              n,
                "m_total":        m_total,
                "m_constr":       m_constr,
                "lp_obj":         lp.get("objective"),
                "lp_t":           lp.get("time"),
                "lp_status":      lp.get("status", "ok"),
                "sdp_obj":        sdp.get("objective"),
                "sdp_t":          sdp.get("time"),
                "sdp_status":     sdp.get("status", "ok"),
                "sdp_prime_obj":  sdp_p.get("objective"),
                "sdp_prime_t":    sdp_p.get("time"),
                "sdp_prime_status": sdp_p.get("status", "ok"),
                "sdp_rlt_obj":    sdp_rlt.get("objective"),
                "sdp_rlt_t":      sdp_rlt.get("time"),
                "sdp_rlt_status": sdp_rlt.get("status", "ok"),
            }

        except Exception as e:
            print(f"⚠️ idx={idx} failed: {e}", flush=True)
            row = {
                "id":             int(getattr(inst, "instance_id", idx) if hasattr(inst, "instance_id") else idx),
                "split":          split,
                "seed":           None,
                "id_global":      f"err_{idx}",
                "group_id":       None,
                "size_tag":       None,
                "bounds_exist":   None,
                "n":              None,
                "m_total":        None,
                "m_constr":       None,
                "lp_obj":         np.nan,
                "lp_t":           np.nan,
                "lp_status":      f"error:{e}",
                "sdp_obj":        np.nan,
                "sdp_t":          np.nan,
                "sdp_status":     f"error:{e}",
                "sdp_prime_obj":  np.nan,
                "sdp_prime_t":    np.nan,
                "sdp_prime_status": f"error:{e}",
                "sdp_rlt_obj":    np.nan,
                "sdp_rlt_t":      np.nan,
                "sdp_rlt_status": f"error:{e}",
            }

        if out_csv:
            pd.DataFrame([row]).to_csv(out_csv, mode="a", index=False, header=not header_written)
            header_written = True

        try:
            del solver
        except Exception:
            pass

        if gc_every and ((idx + 1) % gc_every == 0):
            gc.collect()

        if (idx + 1) % 500 == 0:
            print("MEM_RSS_MB=", proc.memory_info().rss / 1e6, flush=True)

    print("✅ wrote:", out_csv)


# -----------------
# Entry point
# -----------------
if __name__ == "__main__":
    out_csv = "instancesobj_synthetic.csv"

    run_instances_from_pkl(
        pkl_path="instances_synthetic.pkl",
        out_csv=out_csv,
        print_every=50,
        solver_kwargs={
            "use_blocks":      True,
            "block_tol":       1e-8,
            "verbose_blocks":  False,
            "prune_matrices":  False,
            "min_block_edges": 0,
        },
        solve_lp=True,
        solve_sdp=True,
        solve_sdp_prime=True,
        solve_sdp_rlt=True,
    )

Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-17
idx=0 id=0 | n=16 m_constr=48 | LP=-15.727988365800112 | SDP=-5.215732481356499
idx=50 id=50 | n=12 m_constr=26 | LP=-1295.1568378312998 | SDP=-14.03524939270031
idx=100 id=100 | n=19 m_constr=25 | LP=-54.050019845139765 | SDP=-23.55601218344577
idx=150 id=150 | n=4 m_constr=2 | LP=-16.710080275595484 | SDP=-inf
idx=200 id=200 | n=43 m_constr=84 | LP=-24.484006555239556 | SDP=-inf
idx=250 id=250 | n=10 m_constr=12 | LP=-30.228582185633933 | SDP=-30.22858214624881
idx=300 id=300 | n=14 m_constr=22 | LP=-6622.434299590601 | SDP=-365.5849132084524
idx=350 id=350 | n=14 m_constr=28 | LP=-52.19611519825716 | SDP=-18.53010395780297
idx=400 id=400 | n=14 m_constr=22 | LP=-13.021940448919786 | SDP=-inf
idx=450 id=450 | n=5 m_constr=3 | LP=-18.77556773495345 | SDP=-inf
MEM_RSS_MB= 2719.535104
idx=500 id=500 | n=14 m_constr=36 | LP=-10.546919323471027 | SDP=-10.546919322014213
idx=550 id=550 | n=14 m_const

In [4]:
"""import pandas as pd

# Load the saved instances
instances = pd.read_pickle("instances_5_100tr.pkl")

# Choose the instance you want to inspect
index = 0  # Change this to any index you want
inst = instances[index]

print(f"📌 Instance ID: {inst.instance_id}")
print(f"Bounds Exist?: {inst.bounds_exist}\n")"""

'import pandas as pd\n\n# Load the saved instances\ninstances = pd.read_pickle("instances_5_100tr.pkl")\n\n# Choose the instance you want to inspect\nindex = 0  # Change this to any index you want\ninst = instances[index]\n\nprint(f"📌 Instance ID: {inst.instance_id}")\nprint(f"Bounds Exist?: {inst.bounds_exist}\n")'